# 03 — 社区检测与摘要流水线测试

覆盖 `backend/community` 模块的完整流程：社区检测（Phase 1）+ 社区摘要（Phase 2）。

**前置条件**：
- Neo4j 已启动，环境变量已配置
- Neo4j GDS 插件已安装（`graph-data-science`）
- 图谱中已有 `__Entity__` 节点和关系（建议先跑 `01_graph_pipeline.ipynb`）
- LLM API Key 已配置（摘要生成需要）

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.community import CommunityDetectorFactory, CommunitySummarizerFactory
from backend.community.detector import BaseCommunityDetector
from backend.community.summary import BaseSummarizer
from backend.community.summary.base import (
    BaseCommunityDescriber, BaseCommunityRanker, BaseCommunityStorer,
)
from backend.config.settings import community_algorithm, GDS_MEMORY_LIMIT, MAX_WORKERS
from backend.graph.core import get_connection_manager

# 初始化 GDS 和 Neo4j 连接
from graphdatascience import GraphDataScience
from backend.config.neo4jdb import get_db_manager

db = get_db_manager()
graph = db.graph  # Neo4jGraph 实例

gds = GraphDataScience(
    db.neo4j_uri,
    auth=(db.neo4j_username, db.neo4j_password),
)

print(f"社区算法: {community_algorithm}")
print(f"GDS 内存限制: {GDS_MEMORY_LIMIT}GB")
print(f"摘要并行线程: {MAX_WORKERS}")

g:\system1_software\anaconda\envs\mpcrl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


社区算法: leiden
GDS 内存限制: 6GB
摘要并行线程: 4


---
## Phase 1：社区检测

### 1a. 查看图谱现状

In [2]:
print("=" * 50)
print("图谱当前状态")
print("=" * 50)

r = graph.query("MATCH (e:`__Entity__`) RETURN count(e) AS cnt")[0]
entity_count = r['cnt']
print(f"  __Entity__ 节点: {entity_count} 个")

r = graph.query("MATCH ()-[r]->() RETURN count(r) AS cnt")[0]
print(f"  总关系数: {r['cnt']} 条")

r = graph.query("MATCH (c:`__Community__`) RETURN count(c) AS cnt")[0]
print(f"  __Community__ 节点: {r['cnt']} 个（如 >0 表示已有旧社区）")

if entity_count == 0:
    print("  \n  ⚠ 图谱中无 Entity，请先运行 01_graph_pipeline.ipynb 构建知识图谱")

图谱当前状态
  __Entity__ 节点: 37 个
  总关系数: 88 条
  __Community__ 节点: 0 个（如 >0 表示已有旧社区）


### 1b. 创建社区检测器并执行

In [3]:
print("=" * 50)
print(f"[Phase 1] 社区检测（算法: {community_algorithm}）")
print("=" * 50)

detector = CommunityDetectorFactory.create(community_algorithm, gds, graph)
print(f"  检测器: {type(detector).__name__}")

try:
    results = detector.process()
    DETECTION_OK = results['status'] == 'success'
except Exception as e:
    print(f"  ✗ 社区检测失败: {e}")
    DETECTION_OK = False
    results = {}

[Phase 1] 社区检测（算法: leiden）
社区检测参数: CPU=4, 内存=6.0GB, 并发度=4, 节点限制=20000
  检测器: LeidenDetector
开始执行LeidenDetector社区检测...
开始创建社区检测的图投影...
图投影创建成功: 37 节点, 86 关系
开始执行Leiden社区检测...
图包含 2 个连通分量
开始保存Leiden社区检测结果...
社区投影图已清理
图投影清理完成，耗时: 0.14秒


In [4]:
# ── Phase 1 验证 ──
print("=" * 50)
print("[Phase 1 验证] 社区检测结果")
print("=" * 50)

if not DETECTION_OK:
    print("  ⏭ 社区检测未执行，跳过验证")
else:
    # 1. 检测统计
    det = results.get('details', {}).get('detection', {})
    community_count = det.get('communityCount', 0)
    print(f"  ✓ 检测到社区: {community_count} 个")
    
    modularity = det.get('modularity', 0)
    if modularity:
        print(f"  ✓ 模块度: {modularity:.4f}")
    
    component_count = det.get('componentCount', 0)
    if component_count:
        print(f"  ✓ 连通分量: {component_count} 个")
    
    # 2. 性能统计
    perf = results.get('performance', {})
    if perf:
        print(f"  \n  性能:")
        print(f"    投影耗时: {perf.get('projectionTime', 0):.2f}s")
        print(f"    检测耗时: {perf.get('detectionTime', 0):.2f}s")
        print(f"    保存耗时: {perf.get('saveTime', 0):.2f}s")
        print(f"    总耗时:   {perf.get('totalTime', 0):.2f}s")
    
    # 3. 验证 __Community__ 节点已创建
    r = graph.query("""
        MATCH (c:`__Community__`)
        RETURN count(c) AS cnt, count(DISTINCT c.level) AS levels
    """)[0]
    assert r['cnt'] > 0, f"Community 节点数量为 {r['cnt']}"
    print(f"  \n  ✓ __Community__ 节点: {r['cnt']} 个（{r['levels']} 个层级）")
    
    # 4. 验证 IN_COMMUNITY 关系
    r = graph.query("""
        MATCH (e:`__Entity__`)-[:IN_COMMUNITY]->(c:`__Community__`)
        RETURN count(e) AS cnt
    """)[0]
    assert r['cnt'] > 0, f"IN_COMMUNITY 关系为 {r['cnt']}"
    print(f"  ✓ IN_COMMUNITY 关系: {r['cnt']} 条")
    
    # 5. 验证各社区大小分布
    r = graph.query("""
        MATCH (e:`__Entity__`)-[:IN_COMMUNITY]->(c:`__Community__` {level: 0})
        WITH c, count(e) AS size
        RETURN min(size) AS min, max(size) AS max, avg(size) AS avg
    """)[0]
    print(f"  \n  社区大小分布 (level 0):")
    print(f"    最小: {r['min']}, 最大: {r['max']}, 平均: {r['avg']:.1f}")
    
    print(f"\n  ✓ Phase 1 测试通过")
    
    # 保存结果供后续使用
    PHASE1_COMMUNITY_COUNT = community_count

[Phase 1 验证] 社区检测结果
  ✓ 检测到社区: 5 个
  ✓ 模块度: 0.6065
  ✓ 连通分量: 2 个
  
  性能:
    投影耗时: 0.24s
    检测耗时: 0.47s
    保存耗时: 0.61s
    总耗时:   1.46s
  
  ✓ __Community__ 节点: 27 个（3 个层级）
  ✓ IN_COMMUNITY 关系: 37 条
  
  社区大小分布 (level 0):
    最小: 1, 最大: 5, 平均: 2.5

  ✓ Phase 1 测试通过


---
## Phase 2：社区摘要

### 2a. 创建摘要器

In [5]:
print("=" * 50)
print(f"[Phase 2] 社区摘要生成（算法: {community_algorithm}）")
print("=" * 50)

if not DETECTION_OK:
    print("  ⏭ 检测未完成，跳过摘要")
else:
    try:
        summarizer = CommunitySummarizerFactory.create_summarizer(
            community_algorithm, graph
        )
        print(f"  摘要器: {type(summarizer).__name__}")
        SUMMARIZER_OK = True
    except Exception as e:
        print(f"  ✗ 摘要器创建失败: {e}")
        SUMMARIZER_OK = False

[Phase 2] 社区摘要生成（算法: leiden）
社区摘要生成器初始化，并行线程数: 4
  摘要器: LeidenSummarizer


### 2b. 社区排名

In [6]:
if SUMMARIZER_OK:
    print("计算社区排名...")
    summarizer.ranker.calculate_ranks()
    
    r = graph.query("""
        MATCH (c:`__Community__`) WHERE c.community_rank IS NOT NULL
        RETURN count(c) AS cnt
    """)[0]
    print(f"  已排名的社区: {r['cnt']} 个")

计算社区排名...
计算社区权重...
社区权重计算完成，处理了 27 个社区，耗时: 0.43秒
  已排名的社区: 27 个


### 2c. 收集社区信息

In [7]:
if SUMMARIZER_OK:
    print("收集社区详细信息...")
    community_info = summarizer.collect_community_info()
    print(f"  收集到 {len(community_info)} 个社区的信息")
    
    if community_info:
        # 展示第一个社区结构
        c = community_info[0]
        print(f"\n  首个社区 {c.get('communityId', '?')}:")
        print(f"    实体数: {len(c.get('nodes', []))}")
        print(f"    关系数: {len(c.get('rels', []))}")
        
        # 格式化预览
        preview = summarizer.describer.prepare_string(c)
        print(f"\n  格式化预览 (前300字):\n{preview[:300]}")

收集社区详细信息...
收集Leiden社区信息...
找到 15 个Leiden社区，开始收集详细信息
收集到 14 个Leiden社区信息，耗时: 0.57秒
  收集到 14 个社区的信息

  首个社区 0-24:
    实体数: 2
    关系数: 0

  格式化预览 (前300字):
Nodes are:
id: 主轴冷却系统, type: 设备, description: 主轴冷却系统是用于冷却主轴的设备，本次维修对象，包含多个部件。
id: 密封圈, type: 部件, description: 冷却系统中已腐蚀的密封圈，需要更换。

Relationships are:



### 2d. 执行完整摘要生成

In [8]:
if SUMMARIZER_OK:
    print("执行完整社区摘要生成...")
    try:
        summaries = summarizer.process_communities()
        SUMMARY_OK = True
    except Exception as e:
        print(f"  ✗ 摘要生成失败: {e}")
        SUMMARY_OK = False
        summaries = []

执行完整社区摘要生成...
开始处理社区摘要...
计算社区权重...
社区权重计算完成，处理了 27 个社区，耗时: 0.02秒
收集Leiden社区信息...
找到 15 个Leiden社区，开始收集详细信息
收集到 14 个Leiden社区信息，耗时: 0.04秒
开始并行生成 14 个社区摘要，使用 4 个线程...
已处理 10/14 (71.4%)
已处理 14/14 (100.0%)
开始存储 14 个社区摘要...
已存储批次 1/2, 耗时: 0.11秒
已存储批次 2/2, 耗时: 0.01秒

社区摘要处理完成，总耗时: 14.11秒
  社区权重计算: 0.02秒 (0.1%)
  社区信息查询: 0.04秒 (0.3%)
  摘要生成(LLM): 13.93秒 (98.7%)
  结果存储: 0.12秒 (0.9%)


In [9]:
# ── Phase 2 验证 ──
print("=" * 50)
print("[Phase 2 验证] 社区摘要结果")
print("=" * 50)

if not SUMMARIZER_OK:
    print("  ⏭ 摘要未生成，跳过验证")
elif not summaries:
    print("  ⚠ 摘要列表为空（可能社区信息不足）")
else:
    print(f"  ✓ 生成了 {len(summaries)} 个社区摘要")
    
    # 1. 验证摘要结构完整性
    valid = sum(
        1 for s in summaries
        if s.get('community') and s.get('summary')
    )
    print(f"  ✓ 结构完整的摘要: {valid}/{len(summaries)}")
    
    # 2. 查看前 3 个摘要
    for s in summaries[:3]:
        cid = s.get('community', '?')
        text = s.get('summary', '')[:120]
        print(f"\n  [社区 {cid}] {text}...")
    
    if len(summaries) > 3:
        print(f"  ... 还有 {len(summaries) - 3} 个")
    
    # 3. 验证 Neo4j 中已存储
    r = graph.query("""
        MATCH (c:`__Community__`)
        WHERE c.summary IS NOT NULL
        RETURN count(c) AS cnt
    """)[0]
    print(f"\n  ✓ Neo4j 中有摘要的社区: {r['cnt']} 个")
    
    # 4. 验证摘要不为空
    empty = sum(1 for s in summaries if not s.get('summary') or len(s['summary']) < 10)
    if empty:
        print(f"  ⚠ {empty} 个摘要内容过短")
    
    print(f"\n  ✓ Phase 2 测试通过")

[Phase 2 验证] 社区摘要结果
  ✓ 生成了 14 个社区摘要
  ✓ 结构完整的摘要: 14/14

  [社区 0-24] 主轴冷却系统是本次维修的设备，其部件密封圈已腐蚀，需要更换。...

  [社区 0-30] 过滤网堵塞是冷却系统故障的原因之一，原因是过滤网因腐蚀或堵塞影响冷却液流通。过滤网是冷却系统中已腐蚀的部件，需要更换。更换过滤网是维修措施，旨在通过更换已腐蚀的过滤网保证过滤功能。过滤网与过滤网堵塞之间存在关联，即过滤网是堵塞发生的对象。...

  [社区 0-8] 主轴驱动电机过热报警（故障代码E-0123）因冷却单元故障导致温度升至95℃，触发过热保护停机，造成B批次订单延期。维修措施为拆解主轴冷却系统并更换部件，之后进行系统调试以恢复正常运行。...
  ... 还有 11 个

  ✓ Neo4j 中有摘要的社区: 14 个

  ✓ Phase 2 测试通过


---
## Describer / Storer 工具类独立测试

In [10]:
print("=" * 50)
print("[工具类测试] Describer")
print("=" * 50)

# BaseCommunityDescriber
describer = BaseCommunityDescriber()

# 测试数据
test_data = {
    'nodes': [
        {'id': 'entity_1', 'type': 'Equipment', 'description': '数控加工中心'},
        {'id': 'entity_2', 'type': 'Fault', 'description': '主轴过热'},
    ],
    'rels': [
        {'start': 'entity_1', 'type': '发生', 'end': 'entity_2', 'description': '关联'},
    ]
}

result = describer.prepare_string(test_data)
assert 'Nodes are:' in result, "Describer 输出缺少 Nodes 标记"
assert 'Relationships are:' in result, "Describer 输出缺少 Rels 标记"
assert '数控加工中心' in result, "Describer 输出缺少实体描述"
assert '(entity_1)-[:发生]->(entity_2)' in result, "Describer 输出缺少关系"
print(f"  ✓ Describer 格式化正常")
print(f"  输出预览:\n{result}")

# 空数据处理
result_empty = describer.prepare_string({})
assert result_empty, "空数据应返回非空字符串"
print(f"  ✓ Describer 空数据处理正常")

[工具类测试] Describer
  ✓ Describer 格式化正常
  输出预览:
Nodes are:
id: entity_1, type: Equipment, description: 数控加工中心
id: entity_2, type: Fault, description: 主轴过热

Relationships are:
(entity_1)-[:发生]->(entity_2), description: 关联

  ✓ Describer 空数据处理正常


In [11]:
print("=" * 50)
print("[工具类测试] Storer")
print("=" * 50)

storer = BaseCommunityStorer(graph)

# 测试摘要存储
test_summaries = [
    {"community": "test-1", "summary": "测试摘要1", "full_content": "..."},
    {"community": "test-2", "summary": "测试摘要2", "full_content": "..."},
]

storer.store_summaries(test_summaries)

# 验证已存储
r = graph.query("""
    MATCH (c:`__Community__` {id: 'test-1'})
    RETURN c.summary AS summary
""")[0]
assert r['summary'] == '测试摘要1', f"存储验证失败: {r}"
print(f"  ✓ Storer 批量存储正常")

# 空列表
storer.store_summaries([])
print(f"  ✓ Storer 空列表处理正常")

# 清理测试数据
graph.query("MATCH (c:`__Community__`) WHERE c.id STARTS WITH 'test-' DETACH DELETE c")
print(f"  ✓ 测试数据已清理")

[工具类测试] Storer
开始存储 2 个社区摘要...
已存储批次 1/1, 耗时: 0.02秒
  ✓ Storer 批量存储正常
没有社区摘要需要存储
  ✓ Storer 空列表处理正常
  ✓ 测试数据已清理


---
## 全链路汇总

In [12]:
print("\n" + "=" * 50)
print("社区检测最终统计")
print("=" * 50)

r = graph.query("""
    MATCH (c:`__Community__`)
    OPTIONAL MATCH (c)<-[:IN_COMMUNITY]-(e:`__Entity__`)
    RETURN c.id AS community_id, c.level AS level, c.summary IS NOT NULL AS has_summary,
           count(e) AS entity_count, c.community_rank AS rank
    ORDER BY c.level, entity_count DESC
    LIMIT 20
""")
print(f"{'社区ID':<20} {'层级':<6} {'实体数':<8} {'有摘要':<8} {'排名':<8}")
print("-" * 55)
for row in r:
    print(f"{row['community_id']:<20} {str(row.get('level', '?')):<6} "
          f"{row['entity_count']:<8} {str(row['has_summary']):<8} "
          f"{str(row.get('rank', 'N/A')):<8}")

r_total = graph.query("""
    MATCH (c:`__Community__`) RETURN count(c) AS total, 
           count(DISTINCT c.level) AS levels
""")[0]
print(f"\n总计: {r_total['total']} 个社区, {r_total['levels']} 个层级")


社区检测最终统计
社区ID                 层级     实体数      有摘要      排名      
-------------------------------------------------------
0-8                  0      5        True     2       
0-29                 0      4        True     1       
0-26                 0      3        True     1       
0-18                 0      3        True     1       
0-30                 0      3        True     2       
0-17                 0      2        True     2       
0-31                 0      2        True     1       
0-24                 0      2        True     2       
0-21                 0      2        True     1       
0-3                  0      2        True     1       
0-2                  0      2        True     1       
0-19                 0      2        True     1       
0-32                 0      2        True     1       
0-22                 0      2        True     1       
0-4                  0      1        False    1       
1-4                  1      0        False    2       

In [13]:
# ── 全链路测试汇总 ──
print("\n" + "=" * 50)
print("测试汇总")
print("=" * 50)

summary_items = [
    ("Phase 1 社区检测", DETECTION_OK),
    ("Phase 2 摘要生成", SUMMARIZER_OK and len(summaries) > 0 if SUMMARIZER_OK else False),
    ("工具类 Describer", True),
    ("工具类 Storer", True),
]

all_pass = True
for name, passed in summary_items:
    icon = "✓" if passed else "✗"
    if not passed:
        all_pass = False
    print(f"  [{icon}] {name}")

print(f"\n{'全部通过!' if all_pass else '有失败项，请检查上方输出'}")


测试汇总
  [✓] Phase 1 社区检测
  [✓] Phase 2 摘要生成
  [✓] 工具类 Describer
  [✓] 工具类 Storer

全部通过!
